# 01. pdf-inspector 기초 실습

목표: 실제 PDF 파서를 만들지는 않고, pdf-inspector가 왜 `텍스트 연산자`와 `이미지 연산자`를 보고 PDF 라우팅 결정을 내리는지 작은 Python 모델로 이해합니다.

실행 방법: Jupyter Notebook에서 위에서 아래로 실행합니다. 외부 패키지는 필요하지 않습니다.

In [ ]:
from dataclasses import dataclass
from typing import List


@dataclass
class PageSignal:
    page: int
    has_text_operator: bool
    has_image_operator: bool


def inspect_page(page_number: int, content_stream: str) -> PageSignal:
    """PDF content stream에서 핵심 신호만 단순 검사합니다.

    실제 PDF 파서는 압축 해제, 객체 참조, 폰트 인코딩 등을 처리해야 합니다.
    여기서는 학습 목적상 `Tj`/`TJ`는 텍스트 출력, `Do`는 이미지/XObject 출력 신호로 봅니다.
    """
    text_ops = (" Tj", " TJ", "Tj", "TJ")
    image_ops = (" Do", "Do")
    return PageSignal(
        page=page_number,
        has_text_operator=any(op in content_stream for op in text_ops),
        has_image_operator=any(op in content_stream for op in image_ops),
    )


def classify_from_signals(signals: List[PageSignal]) -> dict:
    """페이지별 신호를 문서 유형과 OCR 필요 페이지로 변환합니다."""
    pages_with_text = [s.page for s in signals if s.has_text_operator]
    pages_without_text = [s.page for s in signals if not s.has_text_operator]
    pages_with_images = [s.page for s in signals if s.has_image_operator]

    if len(pages_with_text) == len(signals):
        pdf_type = "TextBased"
    elif len(pages_with_text) == 0 and len(pages_with_images) > 0:
        pdf_type = "Scanned"
    elif len(pages_with_text) == 0:
        pdf_type = "ImageBased"
    else:
        pdf_type = "Mixed"

    confidence = len(pages_with_text) / max(len(signals), 1)
    return {
        "pdf_type": pdf_type,
        "confidence": round(confidence, 3),
        "pages_needing_ocr": pages_without_text,
    }


In [ ]:
documents = {
    "report_text_based": ["BT (Hello) Tj ET", "BT (Table data) TJ ET"],
    "scan_only": ["/Im1 Do", "/Im2 Do"],
    "mixed_contract": ["BT (Clause 1) Tj ET", "/SignatureImage Do", "BT (Appendix) Tj ET"],
}

for name, pages in documents.items():
    signals = [inspect_page(i + 1, stream) for i, stream in enumerate(pages)]
    print(name, classify_from_signals(signals))


해석:

- `TextBased`는 로컬 텍스트 추출과 Markdown 변환을 먼저 시도할 수 있습니다.
- `Scanned`는 텍스트 레이어가 없으므로 OCR 경로가 필요합니다.
- `Mixed`는 문서 전체가 아니라 텍스트가 없는 페이지만 OCR로 보내는 정책이 비용을 줄입니다.